In [32]:
# KMeans Clustering
# Izbor optimalnog broja klastera pomoću Elbow metode i Silhouette analize, kao i vizualizaciju rezultata klasterovanja. 
# Takođe, uključuje kod za čuvanje rezultata klasterovanja u .npy formatu radi efikasnijeg učitavanja i rada sa modelima.

# Ucitavanje potrebnih podataka
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import seaborn as sns
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

datasets = {
    "Scaled dataset": "../../data/reduced/X_scaled.npy",
    "PCA dataset": "../../data/reduced/X_pca.npy"
}

outputDirectory = "../../results/kmeans"

In [33]:
def elbow_method(X, name):
    K = range(2, 15)
    inertia = []

    for k in K:
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(X)
        inertia.append(kmeans.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(K, inertia, marker='o')
    plt.xlabel('Number of clusters')
    plt.ylabel('Inertia')
    plt.title(f'Elbow Method ({name})')
    plt.grid()
    plt.tight_layout()
    plt.savefig(f"{outputDirectory}/elbow_{name}.png", dpi=200)
    plt.close()

    return K[np.argmax(np.diff(inertia, 2)) + 2]

In [34]:
data = pd.read_csv("../../data/preprocessed/preprocessed_dataset.csv")
y = data["expression"].astype("category").cat.codes

results = []
for dataset_name, X_data in datasets.items():
    X = np.load(X_data)

    k = elbow_method(X, dataset_name)
    print(f"Optimal number of clusters for {dataset_name} according to Elbow method: {k}")

    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)

    row = {
        "Dataset": dataset_name,
        "Algorithm": "KMeans",
        "K": k,
        "Silhouette": silhouette_score(X, labels, sample_size=3000),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "ARI": adjusted_rand_score(y, labels),
        "NMI": normalized_mutual_info_score(y, labels)
    }
    results.append(row)

    if dataset_name == "PCA dataset":
        tsne = TSNE(n_components=2, perplexity=30, random_state=42)
        X_tsne = tsne.fit_transform(X)

        plt.figure(figsize=(6,5))
        plt.scatter(X_tsne[:,0], X_tsne[:,1], c=labels, s=5, cmap="tab10")
        plt.title(f"KMeans TSNE ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"{outputDirectory}/kmTSNE_{dataset_name}.png", dpi=200)
        plt.close()

        plt.figure(figsize=(6,5))
        plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, s=5, cmap="tab10")
        plt.title(f"True classes TSNE ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"{outputDirectory}/trueTSNE_{dataset_name}.png", dpi=200)
        plt.close()

        cluster_vs_class = pd.crosstab(
            pd.Series(labels, name="cluster"),
            pd.Series(data["expression"], name="expression"),
            normalize="index"
        )

        # tabela cluster vs true class sa procentima, čuvanje u CSV formatu i štampanje
        cluster_vs_class = cluster_vs_class.round(4)
        cluster_vs_class.to_csv(f"{outputDirectory}/cluster_vs_class_{dataset_name}.csv")
        print(cluster_vs_class)

        cluster_vs_class = pd.crosstab(
            pd.Series(labels, name="cluster"),
            pd.Series(data["expression"], name="expression"),
            normalize="index"
        )

        cluster_vs_class = cluster_vs_class.round(4)
        cluster_vs_class.to_csv(f"{outputDirectory}/cluster_vs_class_{dataset_name}.csv")
        print(cluster_vs_class)

        # heatmap
        plt.figure(figsize=(8,6))
        sns.heatmap(cluster_vs_class, annot=True, cmap="Blues")
        plt.title(f"Cluster vs True Class ({dataset_name})")
        plt.tight_layout()
        plt.savefig(f"{outputDirectory}/cluster_vs_class_heatmap_{dataset_name}.png", dpi=200)
        plt.close()

results_df = pd.DataFrame(results)
results_df = results_df.round({
    "Silhouette": 4,
    "Davies_Bouldin": 4,
    "ARI": 4,
    "NMI": 4
})

results_df.to_csv(f"{outputDirectory}/kmeans_results_summary.csv", index=False)
print(results_df)

Optimal number of clusters for Scaled dataset according to Elbow method: 4
Optimal number of clusters for PCA dataset according to Elbow method: 4
expression  affirmative  conditional  doubt_question  emphasis  negative  \
cluster                                                                    
0                0.0000       0.2032          0.4535    0.1138    0.0143   
1                0.0808       0.1444          0.0674    0.1066    0.0855   
2                0.1182       0.1567          0.0949    0.0985    0.1387   
3                0.0096       0.0592          0.0278    0.0610    0.0870   

expression  relative  topics  wh_question  yn_question  
cluster                                                 
0             0.0415  0.0281       0.0041       0.1415  
1             0.1773  0.1367       0.0978       0.1035  
2             0.0847  0.1485       0.0319       0.1280  
3             0.2780  0.1197       0.2700       0.0877  
expression  affirmative  conditional  doubt_question  